<a href="https://colab.research.google.com/github/roushanSinha10/Multi-modal-agents/blob/main/multimodalagents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Agentic AI refers to artificial intelligence systems that can act autonomously to achieve goals, rather than just responding to prompts. Think of them as AI “agents” that can plan, decide, and take actions—often across multiple steps—without constant human guidance.

####CrewAI is a Python framework designed for building and managing autonomous AI agents that collaborate to complete tasks—kind of like assembling a small “crew” of specialized AI workers that communicate and coordinate with each other.

####CrewAI lets you define:
*   Agents → individual AI roles (e.g., researcher, writer, analyst)
*   Tasks → what each agent is responsible for
*   Crew → the team of agents working together
*   Process flow → how they collaborate (sequential, hierarchical, etc.)
####Instead of one LLM doing everything, CrewAI breaks work into structured multi-agent workflows.

#Single Agent System

#1. Agents

Each agent has:

*   A role (e.g., “Senior Researcher”)
*   Goals
*   Tools (APIs, search, code execution, etc.)
*   Memory (optional)


In [ ]:
##Example:

from crewai import Agent

researcher = Agent(
    role="Researcher",
    goal="Find accurate information about AI trends",
    backstory="Expert in analyzing tech developments",
    llm="gpt-4o-mini",
    verbose=True
)

#2. Tasks

Tasks define what needs to be done.

In [ ]:
from crewai import Task

task = Task(
    description="Write a report on AI agents",
    agent=researcher
)

#3. Crew (the system)

You combine agents and tasks into a working system:

In [ ]:
from crewai import Crew

crew = Crew(
    agents=[researcher],
    tasks=[task]
)

crew.kickoff()

# Multi Agent System

In [ ]:
import os

from crewai import Agent, Task, Crew, Process
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from dotenv import load_dotenv

load_dotenv()

# Set your API keys

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["EXA_API_KEY"] = "YOUR_EXA_API_KEY"

# Optional
EXA_BASE_URL = os.getenv("EXA_BASE_URL")

# EXA Search Tool
exa_search_tool = EXASearchTool(
    api_key=os.environ["EXA_API_KEY"],
    base_url=EXA_BASE_URL
)

# Website Scraper Tool
scrape_website_tool = ScrapeWebsiteTool()

In [ ]:
research_planner = Agent(
    role="Research Planner",
    goal=(
        "Analyze user queries and break them into focused "
        "research objectives and actionable research topics."
    ),
    backstory=(
        "You are an elite research strategist with years of experience "
        "in investigative analysis and enterprise research workflows."
    ),
    verbose=True,
    allow_delegation=True,
    max_rpm=150,
    max_iter=15
)

In [ ]:
researcher = Agent(
    role="Internet Researcher",
    goal=(
        "Conduct deep internet research using search engines "
        "and website analysis tools."
    ),
    backstory=(
        "You are an expert internet researcher skilled in finding "
        "high-quality information from trusted sources."
    ),
    tools=[
        exa_search_tool,
        scrape_website_tool
    ],
    verbose=True,
    allow_delegation=True,
    max_rpm=150,
    max_iter=15
)

In [ ]:
fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify information accuracy, identify inconsistencies, "
        "and detect misinformation."
    ),
    backstory=(
        "You are a professional fact-checking analyst trained "
        "in cross-referencing trusted sources."
    ),
    tools=[
        exa_search_tool,
        scrape_website_tool
    ],
    verbose=True,
    allow_delegation=True,
    max_rpm=150,
    max_iter=15
)

In [ ]:
report_writer = Agent(
    role="Report Writer",
    goal=(
        "Create professional, well-structured, evidence-based reports."
    ),
    backstory=(
        "You are a senior research report writer specializing "
        "in technical writing and analytical communication."
    ),
    verbose=True,
    allow_delegation=False,
    max_rpm=150,
    max_iter=15
)

In [ ]:
create_research_plan_task = Task(
    description=(
        "Analyze the user's query and break it into:\n"
        "- Main research topics\n"
        "- Subtopics\n"
        "- Key research questions\n\n"
        "User Query:\n"
        "{user_query}"
    ),
    expected_output=(
        "A detailed research plan with:\n"
        "- Main research areas\n"
        "- Key questions\n"
        "- Research priorities\n"
        "- Suggested strategy"
    ),
    agent=research_planner
)

In [ ]:
gather_research_data_task = Task(
    description=(
        "Conduct comprehensive internet research using the "
        "research plan.\n\n"
        "Requirements:\n"
        "- Use authoritative sources\n"
        "- Include citations and URLs\n"
        "- Summarize findings clearly"
    ),
    expected_output=(
        "A research dataset containing:\n"
        "- Key findings\n"
        "- Statistics\n"
        "- Citations\n"
        "- URLs\n"
        "- Supporting evidence"
    ),
    agent=researcher
)

In [ ]:
verify_information_quality_task = Task(
    description=(
        "Review all collected research and verify:\n"
        "- Accuracy\n"
        "- Consistency\n"
        "- Source credibility\n"
        "- Potential misinformation"
    ),
    expected_output=(
        "A fact-checking report containing:\n"
        "- Verified claims\n"
        "- Inconsistencies\n"
        "- Reliability assessment\n"
        "- Corrections"
    ),
    agent=fact_checker
)

In [ ]:
write_final_report_task = Task(
    description=(
        "Create a professional final report using verified research.\n\n"
        "The report must include:\n"
        "- Executive summary\n"
        "- Structured sections\n"
        "- Recommendations\n"
        "- Conclusion\n"
        "- Citations"
    ),
    expected_output=(
        "A polished markdown research report with citations "
        "and actionable insights."
    ),
    agent=report_writer
)

In [ ]:
crew = Crew(
    agents=[
        research_planner,
        researcher,
        fact_checker,
        report_writer
    ],
    tasks=[
        create_research_plan_task,
        gather_research_data_task,
        verify_information_quality_task,
        write_final_report_task
    ],
    process=Process.sequential,
    verbose=True
)

In [ ]:
user_query = """
What are the latest trends, risks, and opportunities
in Artificial Intelligence for healthcare in 2025?
"""
result = crew.kickoff(
    inputs={
        "user_query": user_query
    }
)
